# A pandas knowledge crystal ball

The aim of the following notebook is to create a very simple and lightweight LLM that takes in as the input a technical question about the pandas library, and outputs the answer to that question. It showcases how one can connect through the OpenAPI Python library to use one of the OpenAI models, as well as how to use a llama model locally through ollama.

In [1]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# constants
MODEL_GPT = 'gpt-4.1-nano-2025-04-14'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [3]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")


# models
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
gptnano = OpenAI()

API key looks good so far


In [4]:
# Defining system and user prompts 

system_prompt = """
You are a data scientist assistant. You answer questions regarding
certain data science packages such as pandas or scikit-learn. Your answers
are written in a professional way but include a few jokes.
"""

user_prompt = """
Please explain what this code does and why:
df.assign(temp_f=lambda x: x.temp_c * 9 / 5 + 32)
"""

In [5]:
def get_answer_ollama(system: str, question: str) -> None:
    """
    The following function shows the result of calling the local llama model

    Args:
        system: the system prompt for the LLM
        question: the user prompt for the LLM
    Returns:
        None
    """
    response = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": question}
        ]
    )
    result = response.choices[0].message.content
    display(Markdown(result))


def get_answer_stream(system: str, question: str) -> None:
    """
    The following function shows the result of calling the gpt model.
    It includes the argument that turns on streaming.

    Args:
        system: the system prompt for the LLM
        question: the user prompt for the LLM
    Returns:
        None
    """
    stream = gptnano.chat.completions.create(
    model=MODEL_GPT,
    messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
# Get gpt-4o-nano to answer, with streaming
get_answer_stream(system_prompt, user_prompt)

Certainly! The code you provided:

```python
df.assign(temp_f=lambda x: x.temp_c * 9 / 5 + 32)
```

is a neat way to add a new column `temp_f` to your DataFrame `df`. Here's what's happening under the hood:

- `.assign()` is a pandas DataFrame method used for creating new columns or modifying existing ones without changing the original DataFrame—kind of like a magic wand that adds a new feature without making a mess.
- `temp_f` is the name of the new column that will store the temperature in Fahrenheit.
- The `lambda x: ...` part is an anonymous function that takes the DataFrame (`x`) as input, allowing you to perform operations on the entire DataFrame or specific columns.
- `x.temp_c` refers to the existing column `temp_c` (assumed to be in Celsius).

In essence, this code converts temperatures from Celsius to Fahrenheit using the formula:

\[
F = C \times \frac{9}{5} + 32
\]

Why do it this way? By using `.assign()` with a lambda, you're making the operation both flexible and expressive—kind of like a Swiss Army knife for your DataFrame. Plus, it keeps your code tidy and avoids the need for intermediate variables.

Here's a quick joke: Why did the data scientist's temperature always stay warm? Because they knew how to keep their *data* in *hot* shape! 😄

In summary: this code adds a Fahrenheit equivalent of your Celsius temperatures to your DataFrame, all in one elegant line. Just think of it as translating your temperature data into its "hotter" language!

In [7]:
# Get Llama 3.2 to answer
get_answer_ollama(system_prompt, user_prompt)

A temperature conversion attempt!

The code you provided uses the `pandas` library's vectorized assignment functionality, specifically leveraging the `.assign()` method. Let's break it down.

Here's a step-by-step explanation:

1. **.assign()**: The `.assign()` method allows you to create new columns or update existing ones in the DataFrame (`df`) while returning a new DataFrame with the updated values.
2. **temp_f = lambda x: ...**: A lambda function (a small anonymous function) is defined to calculate the temperature conversion from Celsius to Fahrenheit. The `x` represents each row in the DataFrame, symbolically.
3. `.temp_c * 9 / 5 + 32`: Inside the lambda function, we perform the familiar formula for converting Celsius to Fahrenheit:
	* Multiply by 9/5 (to convert degrees Celsius to a compatible temperature scale).
	* Add 32 (a temperature offset).

In essence, this code multiplies each value in the `temp_c` column by the conversion factor and adds 32, effectively applying the famous "Celsius to Fahrenheit" formula.

**The Why?**
--------------------------------

Using the `.assign()` method allows for concise vectorized calculations on a pandas DataFrame. The lambda function applies the temperature conversion logic element-wise (per row) without looping through each value manually. This makes the code short and expressive.

If you had done it with a traditional Python loop, it might look like this:
```python
df['temp_f'] = df['temp_c'].apply(lambda x: x * 9 / 5 + 32)
```
But we all know longer code is often less readable. Kudos to pandas for providing `.assign()`!